[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/cbm-colab-2026/blob/main/notebooks/2-2_%EC%9D%B4%EC%83%81%ED%83%90%EC%A7%80_%EC%A0%95%EC%83%81%EC%9D%84%EB%B0%B0%EC%9A%B0%EA%B3%A0_%EB%B2%97%EC%96%B4%EB%82%A8%EC%B0%BE%EA%B8%B0.ipynb)

# 2-2 · 이상탐지 — 정상 데이터만으로 "평소와 다름"을 찾는다

**⏱ 100분** · 모듈 2: 센서 → 특징 → **① 이상탐지** → 고장진단

## 🎯 이번 시간의 질문

> 고장 데이터가 거의 없을 때, **정상일 때의 데이터만으로** 이상을 알아챌 수 있을까요?

현장 설비는 대부분의 시간 동안 정상입니다. 고장 사례는 드물고, 모든 종류의 고장을 미리 모아 둘 수도 없습니다.
그래서 CBM의 첫 단계는 보통 **이상탐지(Anomaly Detection)**입니다. 원리는 간단합니다.

```
① 정상일 때의 데이터로 "평소 모습"을 배운다  →  ② 새 데이터가 평소에서 얼마나 먼지 점수를 매긴다  →  ③ 점수가 기준을 넘으면 경보
```

**오늘의 상황:** 유압 장치의 **냉각기 감시 시스템**을 만듭니다. 학습에는 **냉각기가 정상(100%)일 때의 사이클만** 씁니다.
그다음, 처음 보는 사이클 336개(냉각기 정상 112 · 효율 저하 112 · 거의 고장 112)에서 경보가 제대로 울리는지 평가합니다.

| 방법 | 원리 | 특징 |
|---|---|---|
| 방법 1 · 통계(z-점수) | 정상의 평균에서 표준편차 몇 배만큼 떨어졌나 | 센서 1개, 계산이 쉽고 설명이 쉬움 |
| 방법 2 · 머신러닝(Isolation Forest) | 정상 무리에서 얼마나 쉽게 떼어낼 수 있나 | 센서 여러 개를 한꺼번에 봄 |

**이번 시간에 익힐 패턴:** P2 조건으로 나누기 · 평균·표준편차로 점수 만들기 · P5 `fit` (정답 없이 학습) · 교차표로 평가 · 겹친 히스토그램

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. (실습 데이터와 한글 글꼴을 내려받습니다)
import os, urllib.request, warnings
REPO = "https://raw.githubusercontent.com/ralbu85/cbm-colab-2026/main/"
for path in ["data/hydraulic_labels.csv",
             "data/hydraulic_features.csv",
             "fonts/NanumGothic.ttf"]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO + path, path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("fonts/NanumGothic.ttf")
plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False,
                     "figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore")
print("준비 완료!")

In [ ]:
# ▶ 그대로 실행하세요
labels = pd.read_csv('data/hydraulic_labels.csv')
features = pd.read_csv('data/hydraulic_features.csv')
data = labels.merge(features, on='cycle_id')
print('읽기 완료:', data.shape)

## 1. 학습용과 평가용 나누기

- `train`(학습용): 기준을 배우는 데 씁니다. 여기서 **냉각기 정상인 사이클만** 골라 `normal`로 둡니다.
- `test`(평가용): 기준을 정한 **뒤에** 성능을 확인하는 데만 씁니다. 학습에 쓴 데이터로 평가하면 성능이 부풀려지기 때문입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
train = data[data['split'] == 'train']
test = data[data['split'] == 'test']
normal = train[train['cooler'] == 100]
print('정상 학습 사이클:', len(normal))
print('평가 사이클:', len(test))
test['cooler'].value_counts()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 정상 학습 사이클 629개로 "평소 모습"을 배우고, 평가 사이클 336개(상태별 112개)로 확인합니다.
참고로 `normal`에는 냉각기만 정상일 뿐, 밸브·펌프·축압기 상태는 여러 가지가 섞여 있습니다. 냉각기 감시 입장에서는 이것도 **평소의 변동**입니다.

---
## 2. 방법 1 · z-점수 (통계적 이상탐지)

2-1에서 냉각기 상태를 가장 잘 보여 준 특징은 오일 온도 `TS1_mean`이었습니다.

$$z = \frac{\text{새 값} - \text{정상의 평균}}{\text{정상의 표준편차}}$$

z = 3이면 "정상 평균에서 표준편차 3개만큼 떨어져 있다"는 뜻입니다. 정상 데이터가 정규분포를 따른다면 |z| > 3인 경우는 약 0.3%로 드뭅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
mu = normal['TS1_mean'].mean()
sigma = normal['TS1_mean'].std()
print('정상 평균:', round(mu, 2), '°C / 정상 표준편차:', round(sigma, 2), '°C')

z = (test['TS1_mean'] - mu) / sigma
z.head()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


**P4 · 겹친 히스토그램** — 냉각기 상태별로 z-점수 분포를 겹쳐 그립니다. `for` 반복문으로 세 상태를 차례로 그립니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
for state in [100, 20, 3]:
    plt.hist(z[test['cooler'] == state], bins=20, alpha=0.6, label=f'냉각기 {state}%')
plt.axvline(3, color='red', linestyle='--', label='기준 z = 3')
plt.xlabel('z-점수')
plt.ylabel('사이클 수')
plt.legend()
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 냉각기 정상(100%)은 z가 0 근처에 모여 있고, 효율 저하(20%)는 z ≈ 12, 거의 고장(3%)은 z ≈ 28로 **정상과 완전히 떨어져** 있습니다.
기준선 z = 3을 어디에 그어도 깔끔하게 나뉩니다.

### 경보 판정과 평가

|z| > 3이면 경보. 실제 상태 × 경보 여부 교차표로 평가합니다. `colnames=['경보']`는 표의 열 제목입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
alarm = z.abs() > 3
pd.crosstab(test['cooler'], alarm, colnames=['경보'])
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


두 가지 성능 지표를 계산합니다(1-2의 탐지율과 같은 개념).
- **탐지율** = 이상(냉각기 20·3%) 중 경보가 울린 비율 → 높을수록 좋음
- **오경보율** = 정상(냉각기 100%) 중 경보가 울린 비율 → 낮을수록 좋음

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
is_fault = test['cooler'] != 100
print('탐지율:', round(alarm[is_fault].mean() * 100, 1), '%')
print('오경보율:', round(alarm[~is_fault].mean() * 100, 1), '%')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 탐지율 100.0%, 오경보율 0.0%. (`~`는 "아니다"라는 뜻입니다. `~is_fault` = 정상)

💬 **결과 해설** — 정상 데이터 629개만으로 만든 기준이 처음 보는 이상 224개를 **모두** 잡고, 정상 112개에는 경보를 **한 번도** 내지 않았습니다.
이렇게 잘 되는 이유는 모델이 뛰어나서가 아니라 **특징이 좋아서**입니다(냉각이 안 되면 기름이 뜨거워지는 직접적인 관계).
그런데 이 센서가 없거나, 신호가 이렇게 뚜렷하지 않다면 어떨까요?

---
## 3. 방법 2 · Isolation Forest (머신러닝 이상탐지)

**상황:** 온도 센서가 고장 나서 쓸 수 없다고 가정합니다. 남은 센서 5개(압력 2, 유량, 모터 전력, 진동)의 평균값을 **한꺼번에** 봐야 합니다.
센서가 여러 개면 "평균에서 몇 σ"를 하나로 정하기 어렵습니다. 이럴 때 머신러닝 이상탐지를 씁니다.

**Isolation Forest의 아이디어** — 데이터 공간에 무작위로 칸막이를 여러 번 쳐서 점 하나를 혼자 남게(고립) 만든다고 생각해 봅시다.
정상 무리 한가운데 있는 점은 칸막이를 아주 많이 쳐야 고립되지만, **무리에서 떨어진 점은 몇 번 만에 고립**됩니다.
"쉽게 고립되는 점일수록 이상"이라고 판단합니다. 이런 무작위 나무 수백 개의 평균으로 점수를 매깁니다.

먼저 센서 두 개로 정상 무리와 평가 데이터를 그려 봅니다.

In [ ]:
# ▶ 그대로 실행하세요
cols = ['PS1_mean', 'PS2_mean', 'FS1_mean', 'EPS1_mean', 'VS1_mean']
plt.figure(figsize=(7, 5))
plt.scatter(normal['PS2_mean'], normal['VS1_mean'], s=10, color='lightgray', label='학습: 정상')
for state in [100, 20, 3]:
    part = test[test['cooler'] == state]
    plt.scatter(part['PS2_mean'], part['VS1_mean'], s=12, label=f'평가: 냉각기 {state}%')
plt.xlabel('압력 PS2 평균 (bar)')
plt.ylabel('진동 VS1 평균 (mm/s)')
plt.legend()
plt.show()

💬 회색(정상 학습 데이터)이 이루는 **무리** 안에 평가용 정상(파랑)이 들어가고, 냉각기 이상(주황·초록)은 무리 **밖**에 있습니다. 회색 무리에서 주황 쪽으로 꼬리처럼 뻗은 점 몇 개는 냉각기를 바꾼 직후 기름이 아직 식지 않은 사이클입니다.
센서 두 개만 보면 경계가 애매한 곳도 있지만, Isolation Forest는 이런 판단을 **센서 5개 공간에서 한꺼번에** 합니다.

### P5 · 모델 학습 — 정답(y) 없이 `fit`

지도학습과 달리 **정답 없이** 정상 데이터(X)만 넣습니다. 그래서 "비지도 학습"이라고 부릅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
from sklearn.ensemble import IsolationForest

iso = IsolationForest(random_state=0)
iso.fit(normal[cols])
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


### 이상 점수와 기준 정하기

`score_samples()`는 **정상다울수록 큰 값**을 줍니다. 앞에 `-`를 붙여 **클수록 이상**한 점수로 바꿉니다.

기준은 어떻게 정할까요? 이상 데이터가 없으니 **정상 데이터의 점수 분포**로 정합니다.
`np.quantile(점수, 0.99)` = 정상 학습 데이터의 99%가 이 값보다 작다 → "정상도 100번 중 1번은 이 기준을 넘는다"는 뜻입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
normal_score = -iso.score_samples(normal[cols])
test_score = -iso.score_samples(test[cols])

threshold = np.quantile(normal_score, 0.99)
print('기준 점수:', round(threshold, 3))

alarm_if = test_score > threshold
pd.crosstab(test['cooler'], alarm_if, colnames=['경보'])
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 기준 점수 약 0.68. 냉각기 20%·3%는 모두 경보(True), 100%는 모두 경보 없음(False).

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
for state in [100, 20, 3]:
    plt.hist(test_score[test['cooler'] == state], bins=20, alpha=0.6, label=f'냉각기 {state}%')
plt.axvline(threshold, color='red', linestyle='--', label='기준 (정상 99%)')
plt.xlabel('이상 점수 (클수록 이상)')
plt.ylabel('사이클 수')
plt.legend()
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 온도 센서 없이도 다른 센서 5개의 **조합**으로 냉각기 이상 224개를 모두 잡고, 오경보는 없었습니다.
냉각기가 나빠지면 온도뿐 아니라 기름의 점도가 바뀌어 압력·진동·모터 전력도 **조금씩** 달라지는데, 하나하나는 작은 변화라도 **여러 센서를 함께 보면** 정상 무리에서 벗어난 것이 드러납니다.

---
## 4. 실습 과제 (40분)

### 📝 과제 1 · 약한 특징 하나로 z-점수 이상탐지 (기준값 실험)
온도 대신 **유량 `FS1_mean` 하나만** 쓸 수 있다면? 2절의 코드에서 특징을 바꾸고, 기준 k = 1, 2, 3에 대해 표를 채우세요.

| 기준 \|z\| > k | 탐지율(%) | 오경보율(%) |
|---|---|---|
| 1 | | |
| 2 | | |
| 3 | | |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: mu2 = normal['FS1_mean'].mean(), sigma2 = ..., z2 = (test['FS1_mean'] - mu2) / sigma2
# for k in [1, 2, 3]: alarm2 = z2.abs() > k  → 탐지율·오경보율 계산


✏️ **나의 답:** k를 키우면 탐지율과 오경보율이 각각 어떻게 변하나요? 여러분이라면 몇으로 정하겠습니까?

### 📝 과제 2 · Isolation Forest의 기준을 바꾸면?
`np.quantile(normal_score, 0.99)`의 0.99를 **0.90, 0.95**로 바꿔 탐지율과 오경보율을 비교하세요.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: for q in [0.90, 0.95, 0.99]: th = np.quantile(normal_score, q); a = test_score > th


✏️ **나의 답:** q를 낮추면 무엇이 달라지나요? 0.90은 "정상 데이터의 몇 %에 경보를 낸다"는 뜻인가요?

### 📝 과제 3 · 펌프 누설도 이상탐지로 잡힐까?
이번에는 **펌프 감시**입니다. 정상 = `pump_leakage == 0`(누설 없음)인 학습 사이클로 Isolation Forest를 학습하고, 평가용에서 누설(1, 2)을 얼마나 잡는지 보세요.
센서는 `cols` 5개를 그대로 쓰고 기준은 0.99로 합니다.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: normal_p = train[train['pump_leakage'] == 0] → iso_p = IsolationForest(random_state=0).fit(normal_p[cols])
# 점수·기준·경보는 3절과 같은 방식. 평가는 pd.crosstab(test['pump_leakage'], alarm_p)


결과가 기대와 다르다면, 누설 상태별 유량 분포를 그려 원인을 찾아보세요.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 2절의 겹친 히스토그램 코드에서 z → test['FS1_mean'], 'cooler' → 'pump_leakage', 상태 [0, 1, 2]


✏️ **나의 답:** 탐지율은 몇 %인가요? 냉각기와 달리 잘 안 되는 이유는 무엇일까요?

### 📝 과제 4 · 생각해 보기
이상탐지 경보가 울렸을 때, 현장 정비원이 **알 수 있는 것**과 **알 수 없는 것**을 한 가지씩 적으세요.

✏️ **나의 답:**

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 이상탐지의 원리 | 정상 데이터로 기준 학습 → 거리/점수 → 기준 넘으면 경보. **고장 데이터가 없어도** 된다 |
| z-점수 | (값 − 정상 평균) ÷ 정상 표준편차. 센서 1개, 설명 쉬움 |
| Isolation Forest | 여러 센서를 함께 보고 "무리에서 떨어진 정도"를 점수로. `fit(X)`만, 정답 없음 |
| 기준 정하기 | 정상 점수의 분위수(예: 99%) 또는 k·σ. 기준을 바꾸면 **탐지율 ↔ 오경보** 교환 |
| 한계 | 미세한 고장은 정상 변동에 묻힌다. 경보는 "무엇이" 고장인지 알려 주지 않는다 → **진단** |

다음: **2-3 · 고장진단 — 어떤 고장인지 분류하는 모델 만들기**